# Data Formats & Storage — 04: Iceberg Advanced II

## What you will learn
This notebook goes deeper into Iceberg's most powerful production features:
1. **Change Data Feed (CDF)** — track every row-level change for CDC pipelines
2. **Table optimization** — `rewrite_data_files`, `rewrite_manifests`
3. **Branching workflow** — isolation between dev/staging/prod
4. **Row-level deletes** — equality vs position deletes, performance impact
5. **Metadata tables** — full observability into your table internals
6. **Multi-table transactions** — atomic writes across tables


In [1]:
import os, time, random, datetime, pathlib, shutil
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

MASTER   = 'spark://spark-master:7077'
DATA_DIR = '/workspace/data'

spark = (
    SparkSession.builder
    .appName('iceberg-advanced-2')
    .master(MASTER)
    .config('spark.executor.memory',            '2g')
    .config('spark.driver.memory',              '1g')
    .config('spark.executor.cores',             '2')
    .config('spark.shuffle.sort.bypassMergeThreshold', '0')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
print(f"Spark {spark.version}")

Spark 4.0.2


## Step 1 — Setup: Sales Table with CDF

We enable the Change Data Feed on creation so every row-level operation
is tracked and queryable.


In [2]:
spark.sql("CREATE DATABASE IF NOT EXISTS local.sales")
spark.sql("DROP TABLE IF EXISTS local.sales.transactions")

spark.sql("""
    CREATE TABLE local.sales.transactions (
        txn_id      BIGINT,
        customer_id BIGINT,
        product     STRING,
        category    STRING,
        amount      DOUBLE,
        status      STRING,
        region      STRING,
        txn_date    DATE
    )
    USING iceberg
    PARTITIONED BY (months(txn_date), region)
    TBLPROPERTIES (
        'write.format.default'            = 'parquet',
        'write.parquet.compression-codec' = 'zstd',
        'write.delete.mode'               = 'merge-on-read',
        'write.update.mode'               = 'merge-on-read',
        'write.merge.mode'                = 'merge-on-read'
    )
""")

random.seed(42)
PRODUCTS  = {"Laptop":("Electronics",999), "Phone":("Electronics",599),
             "Desk":("Furniture",349), "Chair":("Furniture",199),
             "Shirt":("Clothing",49), "Jeans":("Clothing",79),
             "Spark Book":("Books",49), "Python Book":("Books",39)}
REGIONS   = ["EMEA","APAC","AMER","LATAM"]
STATUSES  = ["pending","confirmed","shipped","delivered","cancelled"]

schema = StructType([
    StructField("txn_id",      LongType()),
    StructField("customer_id", LongType()),
    StructField("product",     StringType()),
    StructField("category",    StringType()),
    StructField("amount",      DoubleType()),
    StructField("status",      StringType()),
    StructField("region",      StringType()),
    StructField("txn_date",    DateType()),
])

def make_txns(n, start_id=1, start_date="2024-01-01", days=90):
    base = datetime.date.fromisoformat(start_date)
    rows = []
    for i in range(n):
        prod, (cat, price) = random.choice(list(PRODUCTS.items()))
        d = base + datetime.timedelta(days=random.randint(0, days))
        rows.append((start_id+i, random.randint(1,500), prod, cat,
                     round(price*random.uniform(0.9,1.1),2),
                     random.choice(STATUSES), random.choice(REGIONS), d))
    return rows

batch1 = spark.createDataFrame(make_txns(1000), schema)
batch1.writeTo("local.sales.transactions").append()
print(f"Inserted {spark.table('local.sales.transactions').count():,} transactions")

Inserted 1,000 transactions


## Step 2 — Change Data Feed: Track Every Row Change

Iceberg's CDF records `insert`, `update_preimage`, `update_postimage`,
and `delete` for every row-level operation. This is the foundation for:
- Downstream CDC pipelines
- Audit logs
- Incremental materialised views
- Debugging data quality issues

Unlike Delta CDF, Iceberg CDF works across ALL engines (Trino, Flink, Hive).


In [3]:
# Read changes using Iceberg change log view
# In Iceberg 1.10.1 + Spark 4.0, CDC is read via the .changes metadata table
# which shows all row-level operations between snapshots

# Method 1: .changes metadata table (Iceberg 1.10+)
try:
    changes = spark.sql("""
        SELECT *
        FROM local.sales.transactions.changes
    """)
    print("Changes via .changes metadata table:")
    changes.groupBy("_change_type").count().orderBy("_change_type").show()

    print("Sample updates (before and after):")
    changes.filter(F.col("_change_type").isin("update_preimage","update_postimage")) \
           .select("txn_id","status","_change_type","_change_ordinal") \
           .orderBy("txn_id","_change_ordinal").show(8)

except Exception as e:
    print(f"Note: .changes table not available ({type(e).__name__})")
    print()
    # Method 2: manual snapshot diff — compare two snapshots
    print("Fallback: manual snapshot diff via history + version AS OF")
    snaps = spark.sql("""
        SELECT snapshot_id, committed_at, operation,
               summary['added-records']   AS added,
               summary['deleted-records'] AS deleted
        FROM local.sales.transactions.snapshots
        ORDER BY committed_at
    """).collect()

    print(f"\nAll snapshots ({len(snaps)} total):")
    for s in snaps:
        print(f"  snap={s['snapshot_id']} op={s['operation']} added={s['added']} deleted={s['deleted']}")

    # Compare row counts across snapshots
    if len(snaps) >= 2:
        s_first = snaps[0]["snapshot_id"]
        s_last  = snaps[-1]["snapshot_id"]
        count_first = spark.sql(f"SELECT COUNT(*) FROM local.sales.transactions VERSION AS OF {s_first}").collect()[0][0]
        count_last  = spark.sql(f"SELECT COUNT(*) FROM local.sales.transactions VERSION AS OF {s_last}").collect()[0][0]
        print(f"\nRows at snapshot {s_first}: {count_first:,}")
        print(f"Rows at snapshot {s_last} : {count_last:,}")
        print(f"Net change: {count_last - count_first:+,}")

Changes via .changes metadata table:


+------------+-----+
|_change_type|count|
+------------+-----+
|      INSERT| 1000|
+------------+-----+

Sample updates (before and after):
26/04/16 18:02:17 WARN SparkScanBuilder: Failed to check if _change_type IN ('update_preimage', 'update_postimage') can be pushed down: Cannot find field '_change_type' in struct: struct<1: txn_id: optional long, 2: customer_id: optional long, 3: product: optional string, 4: category: optional string, 5: amount: optional double, 6: status: optional string, 7: region: optional string, 8: txn_date: optional date>
Note: .changes table not available (Py4JJavaError)

Fallback: manual snapshot diff via history + version AS OF

All snapshots (1 total):
  snap=1861313924725612360 op=append added=1000 deleted=None


In [4]:
# Incremental pipeline pattern — using snapshot diff
print("Incremental pipeline — snapshot-based diff:")
print()

snaps = spark.sql("""
    SELECT snapshot_id, committed_at, operation,
           summary['added-records']   AS added,
           summary['deleted-records'] AS deleted,
           summary['total-records']   AS total
    FROM local.sales.transactions.snapshots
    ORDER BY committed_at
""").collect()

# Show what changed per snapshot
for s in snaps:
    added   = int(s["added"]   or 0)
    deleted = int(s["deleted"] or 0)
    total   = int(s["total"]   or 0)
    print(f"  snapshot={s['snapshot_id']} | op={s['operation']:<10} "
          f"added={added:>5} deleted={deleted:>5} total={total:>6}")

print()
print("Revenue distribution (current state):")
spark.table("local.sales.transactions") \
     .groupBy("status") \
     .agg(F.count("*").alias("count"),
          F.sum("amount").alias("total_revenue")) \
     .orderBy(F.desc("total_revenue")) \
     .show()

print("\nIncremental pipeline pattern:")
print("  1. Record last_processed_snapshot at job start")
print("  2. Read data VERSION AS OF last_snapshot → compute baseline")
print("  3. Read current table → compute current state")
print("  4. Diff = current - baseline (detect changes)")
print("  5. Update checkpoint to current snapshot_id")

Incremental pipeline — snapshot-based diff:

  snapshot=1861313924725612360 | op=append     added= 1000 deleted=    0 total=  1000

Revenue distribution (current state):


[Stage 17:>                                                         (0 + 3) / 3]

+---------+-----+-----------------+
|   status|count|    total_revenue|
+---------+-----+-----------------+
|cancelled|  212|         66779.78|
|delivered|  203|         63349.68|
|confirmed|  208|59452.91000000002|
|  shipped|  195|          54119.3|
|  pending|  182|         48690.29|
+---------+-----+-----------------+


Incremental pipeline pattern:
  1. Record last_processed_snapshot at job start
  2. Read data VERSION AS OF last_snapshot → compute baseline
  3. Read current table → compute current state
  4. Diff = current - baseline (detect changes)
  5. Update checkpoint to current snapshot_id


## Step 3 — Branching Workflow: Dev / Staging / Prod

Iceberg branches let you work on isolated copies of your table data
without duplicating files. The branch is just a pointer to a snapshot.

**Use case:** test a risky transformation on a `staging` branch,
validate it, then fast-forward `main` to the staging snapshot.


In [5]:
# Tag current production state
spark.sql("""
    ALTER TABLE local.sales.transactions
    CREATE TAG prod_2024_q1 RETAIN 30 DAYS
""")

# Create staging branch for experimental ETL
spark.sql("""
    ALTER TABLE local.sales.transactions
    CREATE BRANCH staging RETAIN 7 DAYS
""")

print("Branches and tags:")
spark.sql("SELECT name, type, snapshot_id FROM local.sales.transactions.refs") \
     .show(truncate=False)

# Apply risky transformation on staging branch ONLY
spark.sql("""
    UPDATE local.sales.transactions.branch_staging
    SET amount = amount * 1.15
    WHERE region = 'EMEA' AND txn_date >= '2024-03-01'
""")

# Compare main vs staging
main_rev    = spark.table("local.sales.transactions") \
                   .filter(F.col("region")=="EMEA") \
                   .agg(F.sum("amount")).collect()[0][0]
staging_rev = spark.sql("""
    SELECT SUM(amount)
    FROM local.sales.transactions VERSION AS OF 'staging'
    WHERE region = 'EMEA'
""").collect()[0][0]

print(f"\nEMEA revenue on main    : {main_rev:>12,.2f}")
print(f"EMEA revenue on staging  : {staging_rev:>12,.2f}")
print(f"Difference (15% uplift)  : {staging_rev-main_rev:>12,.2f}")
print()
print("Staging branch validated — safe to fast-forward main.")

Branches and tags:
+------------+------+-------------------+
|name        |type  |snapshot_id        |
+------------+------+-------------------+
|staging     |BRANCH|1861313924725612360|
|main        |BRANCH|1861313924725612360|
|prod_2024_q1|TAG   |1861313924725612360|
+------------+------+-------------------+


EMEA revenue on main    :    83,312.23
EMEA revenue on staging  :    87,521.19
Difference (15% uplift)  :     4,208.96

Staging branch validated — safe to fast-forward main.


In [6]:
# If staging looks good: fast-forward main to staging
# In production this would be: ALTER TABLE ... SET BRANCH main TO staging_snapshot_id
staging_snap = spark.sql("""
    SELECT snapshot_id FROM local.sales.transactions.refs
    WHERE name = 'staging'
""").collect()[0]["snapshot_id"]

print(f"Staging snapshot ID: {staging_snap}")
print("In production: CALL catalog.system.fast_forward('table', 'main', 'staging')")
print("This atomically moves main to point at the staging snapshot.")
print()
print("If staging was BAD: just drop the branch — no cleanup needed.")
spark.sql("ALTER TABLE local.sales.transactions DROP BRANCH staging")
print("Staging branch dropped — main is unchanged.")
spark.sql("SELECT name, type FROM local.sales.transactions.refs").show()

Staging snapshot ID: 6163851914666519457
In production: CALL catalog.system.fast_forward('table', 'main', 'staging')
This atomically moves main to point at the staging snapshot.

If staging was BAD: just drop the branch — no cleanup needed.
Staging branch dropped — main is unchanged.
+------------+------+
|        name|  type|
+------------+------+
|        main|BRANCH|
|prod_2024_q1|   TAG|
+------------+------+



## Step 4 — Merge-on-Read vs Copy-on-Write

Iceberg supports two delete/update strategies:

| Strategy | Write cost | Read cost | Best for |
|---|---|---|---|
| **Copy-on-Write (CoW)** | High — rewrites data files | Low — clean files | Read-heavy workloads |
| **Merge-on-Read (MoR)** | Low — writes delete files | Higher — merges at read | Write-heavy, frequent updates |

MoR writes small **delete files** (positional or equality) instead of
rewriting data files. At read time, Spark merges data files with delete files.
After accumulation, run `rewrite_data_files` to compact back to CoW layout.


In [7]:
# Show current delete files (from our earlier UPDATE/DELETE)
print("Data files vs delete files:")
spark.sql("""
    SELECT content,
           COUNT(*)                            AS file_count,
           SUM(record_count)                   AS total_records,
           SUM(file_size_in_bytes)/1024/1024   AS total_mb
    FROM local.sales.transactions.files
    GROUP BY content
""").show()

print("Content codes: 0=DATA, 1=POSITION_DELETES, 2=EQUALITY_DELETES")
print()
print("Many delete files → high read amplification → time to compact")
print()

# Compact: merge delete files back into data files
print("Running rewrite_data_files to apply deletes...")
spark.sql("""
    CALL local.system.rewrite_data_files(
        table    => 'local.sales.transactions',
        strategy => 'binpack',
        options  => map(
            'target-file-size-bytes', '134217728',
            'delete-file-threshold',  '2'
        )
    )
""").show()

print("After compaction:")
spark.sql("""
    SELECT content, COUNT(*) AS files, SUM(record_count) AS records
    FROM local.sales.transactions.files
    GROUP BY content
""").show()

Data files vs delete files:
+-------+----------+-------------+-------------------+
|content|file_count|total_records|           total_mb|
+-------+----------+-------------+-------------------+
|      0|        12|         1000|0.04164695739746094|
+-------+----------+-------------+-------------------+

Content codes: 0=DATA, 1=POSITION_DELETES, 2=EQUALITY_DELETES

Many delete files → high read amplification → time to compact

Running rewrite_data_files to apply deletes...
+--------------------------+----------------------+---------------------+-----------------------+--------------------------+
|rewritten_data_files_count|added_data_files_count|rewritten_bytes_count|failed_data_files_count|removed_delete_files_count|
+--------------------------+----------------------+---------------------+-----------------------+--------------------------+
|                         0|                     0|                    0|                      0|                         0|
+----------------------

## Step 5 — Metadata Tables: Full Observability

Iceberg exposes rich metadata through virtual tables.
These are invaluable for debugging, capacity planning, and auditing.


In [8]:
print("Available Iceberg metadata tables:")
metadata_tables = {
    "snapshots":  "SELECT snapshot_id, committed_at, operation, summary['total-records'] AS records FROM local.sales.transactions.snapshots ORDER BY committed_at",
    "files":      "SELECT content, COUNT(*) files, SUM(record_count) records, AVG(file_size_in_bytes)/1024 avg_kb FROM local.sales.transactions.files GROUP BY content",
    "manifests":  "SELECT partition_spec_id, length/1024 kb, added_data_files_count, existing_data_files_count, added_delete_files_count FROM local.sales.transactions.manifests LIMIT 5",
    "history":    "SELECT made_current_at, snapshot_id, is_current_ancestor FROM local.sales.transactions.history ORDER BY made_current_at",
    "partitions": "SELECT partition, record_count, file_count FROM local.sales.transactions.partitions ORDER BY record_count DESC LIMIT 8",
}

for table, query in metadata_tables.items():
    print(f"\n--- {table} ---")
    spark.sql(query).show(truncate=False)

Available Iceberg metadata tables:

--- snapshots ---
+-------------------+-----------------------+---------+-------+
|snapshot_id        |committed_at           |operation|records|
+-------------------+-----------------------+---------+-------+
|1861313924725612360|2026-04-16 18:02:13.265|append   |1000   |
|6163851914666519457|2026-04-16 18:02:21.697|overwrite|1090   |
+-------------------+-----------------------+---------+-------+


--- files ---
+-------+-----+-------+------------------+
|content|files|records|avg_kb            |
+-------+-----+-------+------------------+
|0      |12   |1000   |3.5538736979166665|
+-------+-----+-------+------------------+


--- manifests ---
+-----------------+------------+----------------------+-------------------------+------------------------+
|partition_spec_id|kb          |added_data_files_count|existing_data_files_count|added_delete_files_count|
+-----------------+------------+----------------------+-------------------------+----------------

## Step 6 — Table-Level Statistics and Query Optimization

Iceberg collects per-column statistics (min/max/null count) at write time.
These enable **data skipping** — Spark skips files where the filter
cannot possibly match based on min/max values.


In [9]:
# Observe data skipping in action
print("Query without partition filter (full scan):")
t0 = time.time()
r1 = spark.table("local.sales.transactions") \
          .filter(F.col("category") == "Electronics") \
          .agg(F.sum("amount"), F.count("*")).collect()
t1 = time.time() - t0
print(f"  Result: {r1[0]}  Time: {t1:.3f}s")

print("\nQuery with partition filter (partition pruning):")
t0 = time.time()
r2 = spark.table("local.sales.transactions") \
          .filter((F.col("region") == "AMER") &
                  (F.col("txn_date") >= datetime.date(2024,1,1)) &
                  (F.col("txn_date") <  datetime.date(2024,4,1)) &
                  (F.col("category") == "Electronics")) \
          .agg(F.sum("amount"), F.count("*")).collect()
t2 = time.time() - t0
print(f"  Result: {r2[0]}  Time: {t2:.3f}s")
print(f"  Partition pruning speedup: {t1/t2:.1f}x")

# Show partition stats
print("\nPartition layout:")
spark.sql("""
    SELECT spec_id,
           partition.region      AS region,
           record_count,
           file_count,
           total_data_file_size_in_bytes/1024 AS total_kb
    FROM local.sales.transactions.partitions
    ORDER BY record_count DESC
    LIMIT 10
""").show()

Query without partition filter (full scan):
  Result: Row(sum(amount)=198268.96000000002, count(1)=245)  Time: 0.454s

Query with partition filter (partition pruning):
  Result: Row(sum(amount)=48391.45000000002, count(1)=60)  Time: 0.519s
  Partition pruning speedup: 0.9x

Partition layout:
+-------+------+------------+----------+------------+
|spec_id|region|record_count|file_count|    total_kb|
+-------+------+------------+----------+------------+
|      0| LATAM|         102|         1|3.6982421875|
|      0|  EMEA|          90|         1|  3.58984375|
|      0|  AMER|          89|         1|  3.58984375|
|      0|  EMEA|          87|         1|  3.55859375|
|      0|  AMER|          87|         1|3.5439453125|
|      0|  APAC|          86|         1|3.6083984375|
|      0|  APAC|          84|         1|  3.58203125|
|      0|  AMER|          84|         1| 3.595703125|
|      0| LATAM|          77|         1|   3.5078125|
|      0|  APAC|          76|         1|  3.49609375|
+----

## Summary

### Iceberg Advanced II — Key Takeaways

| Feature | Command | When to use |
|---|---|---|
| Change Data Feed | `load("table#changes")` with snapshot range | CDC, incremental pipelines |
| Branching | `CREATE BRANCH name` | Isolated ETL testing |
| Fast-forward | `CALL system.fast_forward(...)` | Promote staging to prod |
| Tagging | `CREATE TAG name` | Reproducibility checkpoints |
| MoR compaction | `CALL system.rewrite_data_files(...)` | After many small updates |
| Data skipping | Partition filters + column stats | Always — free performance |
| Metadata tables | `table.snapshots`, `.files`, `.partitions` | Observability, debugging |

**Next:** `05_delta_advanced_2.ipynb` — Liquid Clustering, DML optimizations.
